# player_stance 3-class 분류기 학습 (Phase 5)

**모델:** `klue/roberta-large` fine-tuning (변경 가능)
**분류:** 3-class CrossEntropyLoss (negative / neutral / positive)
**입력:** title (seq-A) + query_player + description_snippet (seq-B)
**⚠️ event_summary 입력 제거** — Phase 5-C: train/infer 정합성 확보
**max_length:** 256
**목표:** val macro F1 ≥ 0.75
**예상 소요:** T4 GPU 기준 약 20~35분

**사전 준비**
- 런타임 유형: T4 GPU (런타임 → 런타임 유형 변경)
- 업로드 파일: `labeled_players.csv`

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 수 시간 소요')

In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# ── 하이퍼파라미터 (필요 시 수정) ─────────────────────────────────────────────
PRETRAINED   = 'klue/roberta-large'   # 대안: 'monologg/koelectra-small-v3-discriminator'
MAX_LENGTH   = 256
BATCH_SIZE   = 8
GRAD_ACCUM   = 2          # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
EPOCHS       = 5
LR           = 3e-5
WARMUP_RATIO = 0.1
SEED         = 42
VAL_SPLIT    = 0.15
SNIPPET_LEN  = 300
DATA_DIR     = '/content/data'
OUTPUT_DIR   = '/content/player_stance_model'

STANCE_LABELS = ['negative', 'neutral', 'positive']
LABEL2ID = {l: i for i, l in enumerate(STANCE_LABELS)}
ID2LABEL = {i: l for i, l in enumerate(STANCE_LABELS)}
print('Config 설정 완료')

In [ ]:
import os
from google.colab import files

os.makedirs(DATA_DIR, exist_ok=True)
print('labeled_players.csv 를 선택하세요.')
uploaded = files.upload()
for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장: {dst}  ({len(content):,} bytes)')

In [ ]:
import pandas as pd

path = f'{DATA_DIR}/labeled_players.csv'
if not os.path.exists(path):
    raise FileNotFoundError('labeled_players.csv 를 업로드하세요.')

df = pd.read_csv(path, encoding='utf-8-sig')
df_lbl = df.dropna(subset=['player_stance'])
df_lbl = df_lbl[df_lbl['player_stance'].isin(STANCE_LABELS)]

print(f'labeled_players.csv: 총 {len(df)}행 → 학습 가능 {len(df_lbl)}행 (제외 {len(df)-len(df_lbl)}행)')
print('\nplayer_stance 분포:')
for l in STANCE_LABELS:
    cnt = (df_lbl['player_stance'] == l).sum()
    print(f'  {l}: {cnt} ({cnt/max(len(df_lbl),1):.1%})')

print(f'\nquery_player 분포 (상위 20):')
print(df_lbl['query_player'].value_counts().head(20).to_string())

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


class PlayerStanceDataset(Dataset):
    """seq-A: title  /  seq-B: query_player + description_snippet (event_summary 제외)"""
    def __init__(self, titles, player_snippets, labels, tokenizer):
        self.encodings = tokenizer(
            titles, player_snippets,
            truncation='only_second', padding='max_length',
            max_length=MAX_LENGTH, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def load_data():
    path = Path(DATA_DIR) / 'labeled_players.csv'
    df = pd.read_csv(path, encoding='utf-8-sig')

    required = {'title', 'player_stance', 'query_player'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'필수 컬럼 없음: {missing}')

    before = len(df)
    df = df.dropna(subset=['player_stance'])
    df = df[df['player_stance'].isin(STANCE_LABELS)].copy()
    print(f'labeled_players.csv: {len(df)}행 로드 (전체 {before}행 중 {before-len(df)}행 제외)')

    df['title'] = df['title'].fillna('').astype(str).str.strip()
    df['query_player'] = df['query_player'].fillna('').astype(str).str.strip()
    df['description_snippet'] = (
        df['description_snippet'].fillna('').astype(str)
        .str[:SNIPPET_LEN].str.strip()
    )
    # seq-B: 선수명(앵커) + 기사 snippet
    # event_summary는 인퍼런스 시점에 존재하지 않으므로 제외 (Phase 5-C)
    df['player_snippet'] = (df['query_player'] + ' ' + df['description_snippet']).str.strip()

    # 동일 (title + player) 쌍에서 다른 stance → 제거
    df['_label_id'] = df['player_stance'].map(LABEL2ID)
    conflict_key = df['title'] + '|||' + df['query_player']
    df_tmp = df.assign(_key=conflict_key)
    conflicting = (
        df_tmp.groupby('_key')['_label_id'].nunique()
        .loc[lambda c: c > 1].index
    )
    if len(conflicting):
        mask = conflict_key.isin(conflicting)
        print(f'  DROP {mask.sum()}행 ((title+player) 충돌 쌍)')
        df = df[~mask].reset_index(drop=True)

    df = df.drop_duplicates(subset=['title', 'query_player']).reset_index(drop=True)

    labels = df['player_stance'].map(LABEL2ID).tolist()
    print(f'\n최종 데이터셋: {len(labels)}행')
    for l in STANCE_LABELS:
        print(f'  {l}: {labels.count(LABEL2ID[l])}')
    return df['title'].tolist(), df['player_snippet'].tolist(), labels


def compute_class_weights(labels):
    counts = np.bincount(labels, minlength=len(STANCE_LABELS)).astype(float)
    if np.any(counts == 0):
        missing = [STANCE_LABELS[i] for i, c in enumerate(counts) if c == 0]
        raise ValueError(f'학습 분할에 클래스 없음: {missing}')
    weights = counts.sum() / (len(STANCE_LABELS) * counts)
    weights = np.clip(weights, 0.3, 5.0)
    print(f'Class weights: {dict(zip(STANCE_LABELS, weights.round(3).tolist()))}')
    return torch.tensor(weights, dtype=torch.float)


print('클래스 및 함수 정의 완료')

In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_gpu = device.type == 'cuda'
    print(f'Device: {device}\n')

    titles, player_snippets, labels = load_data()

    tr_t, va_t, tr_ps, va_ps, tr_l, va_l = train_test_split(
        titles, player_snippets, labels,
        test_size=VAL_SPLIT, random_state=SEED, stratify=labels,
    )
    print(f'Train: {len(tr_t)}  Val: {len(va_t)}\n')

    print(f'모델 로드: {PRETRAINED} ...')
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED,
        num_labels=len(STANCE_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(device)

    nw = 2 if use_gpu else 0
    train_ds = PlayerStanceDataset(tr_t, tr_ps, tr_l, tokenizer)
    val_ds   = PlayerStanceDataset(va_t, va_ps, va_l, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=use_gpu)
    val_loader   = DataLoader(val_ds,   batch_size=16,         shuffle=False, num_workers=nw, pin_memory=use_gpu)

    class_weights = compute_class_weights(tr_l).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps_per_epoch = max(len(train_loader) // GRAD_ACCUM, 1)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * WARMUP_RATIO), total_steps,
    )

    best_f1 = 0.0; best_epoch = 0
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader, 1):
            lbl = batch.pop('labels').to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = loss_fn(model(**batch).logits, lbl) / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                lbl = batch.pop('labels')
                batch = {k: v.to(device) for k, v in batch.items()}
                preds = model(**batch).logits.argmax(dim=-1).cpu().tolist()
                all_preds.extend(preds)
                all_true.extend(lbl.tolist())

        macro_f1 = f1_score(all_true, all_preds, average='macro', zero_division=0)
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  macro_f1={macro_f1:.4f}')

        if best_epoch == 0 or macro_f1 > best_f1:
            best_f1 = macro_f1; best_epoch = epoch
            model.save_pretrained(OUTPUT_DIR)
            tokenizer.save_pretrained(OUTPUT_DIR)
            with open(f'{OUTPUT_DIR}/player_stance_config.json', 'w', encoding='utf-8') as f:
                json.dump({'labels': STANCE_LABELS, 'label2id': LABEL2ID, 'id2label': ID2LABEL}, f, indent=2, ensure_ascii=False)
            print('  → Best checkpoint 저장')

    print(f'\n학습 완료 — best macro_f1={best_f1:.4f} (epoch {best_epoch})')

    # Best checkpoint 최종 리포트
    model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device).eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in val_loader:
            lbl = batch.pop('labels')
            batch = {k: v.to(device) for k, v in batch.items()}
            all_preds.extend(model(**batch).logits.argmax(dim=-1).cpu().tolist())
            all_true.extend(lbl.tolist())
    print('\nClassification report (best checkpoint):')
    print(classification_report(all_true, all_preds, target_names=STANCE_LABELS, zero_division=0))
    return best_f1


best_f1 = train()

In [ ]:
print('=== 저장된 파일 ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f:<40} {size:>10,} bytes')

cfg = json.load(open(f'{OUTPUT_DIR}/player_stance_config.json'))
print(f'\nlabel2id: {cfg["label2id"]}')
status = '✓ 달성' if best_f1 >= 0.75 else '✗ 미달 — epoch 증가 또는 lr 조정 고려'
print(f'목표 macro_f1 >= 0.75  {status}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
_mdl = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).eval()

def predict_player_stance(title, player, snippet=''):
    # seq-B: 선수명 + snippet (event_summary 없음 — 인퍼런스 시점과 동일)
    player_snippet = f'{player} {snippet[:SNIPPET_LEN]}'.strip()
    enc = _tok(title, player_snippet, truncation='only_second',
               padding='max_length', max_length=MAX_LENGTH, return_tensors='pt')
    with torch.no_grad():
        probs = torch.softmax(_mdl(**enc).logits[0], dim=-1).tolist()
    best_idx = max(range(len(probs)), key=lambda i: probs[i])
    return {'label': STANCE_LABELS[best_idx], 'conf': round(probs[best_idx], 4)}

# (title, player, snippet, expected)
TEST_CASES = [
    ('나균안, 7이닝 무실점 완벽 투구', '나균안', '나균안이 7이닝 무실점으로 팀 승리를 이끌었다.', 'positive'),
    ('롯데 불펜진 붕괴…박세웅 9실점 대패', '박세웅', '박세웅이 9실점을 허용하며 팀 패배 원인이 됐다.', 'negative'),
    ('전준우, 내일 삼성전 4번 타자 선발 예고', '전준우', '롯데는 전준우를 내일 4번 타자로 기용한다.', 'neutral'),
    ('롯데 자이언츠, 삼성 7대3 대승', '이민석', '이민석이 결승 홈런·3타점으로 팀 승리를 견인했다.', 'positive'),
    ('나균안 부상으로 1군 엔트리 말소', '나균안', '나균안이 부상으로 말소됐다.', 'negative'),
]

print('=== Smoke Test (event_summary 없이 예측) ===')
passed = 0
for title, player, snippet, expected in TEST_CASES:
    r = predict_player_stance(title, player, snippet)
    ok = r['label'] == expected
    passed += int(ok)
    mark = 'O' if ok else 'X'
    print(f'  {mark} [{r["label"]:>8}] conf={r["conf"]:.3f}  (기대: {expected})  [{player}]')
    print(f'     {title}')
print(f'\n결과: {passed}/{len(TEST_CASES)} 통과')

In [ ]:
import shutil, zipfile
from google.colab import files

ZIP = '/content/player_stance_model.zip'
shutil.make_archive('/content/player_stance_model', 'zip', OUTPUT_DIR)
print(f'압축 완료: {ZIP}')
with zipfile.ZipFile(ZIP) as z:
    for name in sorted(z.namelist()):
        print(f'  {name:<45} {z.getinfo(name).file_size:>10,} bytes')
files.download(ZIP)

## 다운로드 후 로컬 배치

```
player_stance_model.zip 압축 해제
  → training/models/player_stance_koelectra/
```

필수 파일:
- `config.json`, `model.safetensors` (또는 `pytorch_model.bin`)
- `tokenizer_config.json`, `vocab.txt`
- `player_stance_config.json` ← 라벨 매핑

환경변수: `PLAYER_STANCE_CLASSIFIER_MODEL_DIR=/app/models/player_stance_koelectra`

**인퍼런스 입력 구조 (Phase 5-C 확정):**
- seq-A: `title`
- seq-B: `player_name + " " + description_snippet`
- `event_summary` **사용 안 함** — `backend/models/player_stance_classifier.py`와 동일